In [ ]:
# ============================================================
# MAMBA EXPERIMENT — KAGGLE ENVIRONMENT SETUP
# ============================================================

import os
import sys
import subprocess
import torch

print("=" * 70)
print("MAMBA EXPERIMENT — ENVIRONMENT SETUP")
print("=" * 70)

# ============================================================
# KAGGLE DATASET ROOTS
# ============================================================

TRAINING_PREP_ROOT = (
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

FINAL_RECOVERY_ROOT = (
    "/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"
)

print("Training-prep dataset:")
print(TRAINING_PREP_ROOT)

print()
print("Final-recovery dataset:")
print(FINAL_RECOVERY_ROOT)

# ============================================================
# VERIFY DATASETS ARE ATTACHED
# ============================================================

if not os.path.isdir(TRAINING_PREP_ROOT):
    raise FileNotFoundError(
        f"""
Training-prep dataset was not found:

{TRAINING_PREP_ROOT}

Make sure the Kaggle dataset
'oipranesh/cs781-phase4-training-prep'
is added to this notebook.
"""
    )

if not os.path.isdir(FINAL_RECOVERY_ROOT):
    raise FileNotFoundError(
        f"""
Final-recovery dataset was not found:

{FINAL_RECOVERY_ROOT}

Make sure the Kaggle dataset
'oipranesh/cs781-phase4-final-recovery'
is added to this notebook.
"""
    )

print("✓ Training-prep dataset mounted")
print("✓ Final-recovery dataset mounted")

# ============================================================
# PYTORCH / GPU
# ============================================================

print()
print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for the Mamba experiment."
    )

DEVICE = torch.device("cuda:0")

print("GPU:", torch.cuda.get_device_name(DEVICE))

gpu_memory = (
    torch.cuda.get_device_properties(DEVICE).total_memory
    / 1024**3
)

print(
    "GPU memory:",
    f"{gpu_memory:.2f} GB"
)

print()
print("=" * 70)
print("✓ BASE ENVIRONMENT VERIFIED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — DATASET ARTIFACT DISCOVERY
# ============================================================

import os

def show_tree(root, max_depth=3):

    root = os.path.abspath(root)

    print()
    print("=" * 70)
    print(root)
    print("=" * 70)

    root_depth = root.rstrip(os.sep).count(os.sep)

    for current_root, dirs, files in os.walk(root):

        depth = (
            current_root.rstrip(os.sep).count(os.sep)
            - root_depth
        )

        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "  " * depth

        print(f"{indent}{os.path.basename(current_root)}/")

        for filename in sorted(files):
            path = os.path.join(current_root, filename)
            size_gb = os.path.getsize(path) / 1024**3

            print(
                f"{indent}  {filename}"
                f"  ({size_gb:.3f} GB)"
            )


show_tree(
    TRAINING_PREP_ROOT,
    max_depth=3,
)

show_tree(
    FINAL_RECOVERY_ROOT,
    max_depth=3,
)

In [ ]:
# ============================================================
# MAMBA — RESOLVE CANONICAL ARTIFACT PATHS
# ============================================================

def find_file(root, filename):

    matches = []

    for current_root, dirs, files in os.walk(root):

        if filename in files:
            matches.append(
                os.path.join(
                    current_root,
                    filename
                )
            )

    return matches


# ------------------------------------------------------------
# TRAINING MANIFEST
# ------------------------------------------------------------

train_manifest_matches = find_file(
    TRAINING_PREP_ROOT,
    "train_manifest.jsonl",
)

if len(train_manifest_matches) != 1:

    raise RuntimeError(
        "Expected exactly one train_manifest.jsonl, "
        f"found {len(train_manifest_matches)}:\n"
        + "\n".join(train_manifest_matches)
    )

TRAIN_MANIFEST = train_manifest_matches[0]


# ------------------------------------------------------------
# TOKENIZER
# ------------------------------------------------------------

tokenizer_matches = find_file(
    TRAINING_PREP_ROOT,
    "legal_bpe_32k.json",
)

if len(tokenizer_matches) != 1:

    raise RuntimeError(
        "Expected exactly one legal_bpe_32k.json, "
        f"found {len(tokenizer_matches)}:\n"
        + "\n".join(tokenizer_matches)
    )

TOKENIZER_PATH = tokenizer_matches[0]


# ------------------------------------------------------------
# SOURCE INDEX
# ------------------------------------------------------------

index_matches = find_file(
    FINAL_RECOVERY_ROOT,
    "source_index.jsonl",
)

if len(index_matches) == 0:

    print(
        "⚠ source_index.jsonl was not found "
        "inside final-recovery."
    )

    INDEX_PATH = None

elif len(index_matches) == 1:

    INDEX_PATH = index_matches[0]

else:

    raise RuntimeError(
        "Multiple source_index.jsonl files found:\n"
        + "\n".join(index_matches)
    )


# ============================================================
# REPORT
# ============================================================

print("=" * 70)
print("CANONICAL MAMBA ARTIFACT PATHS")
print("=" * 70)

print()
print("TRAIN_MANIFEST")
print(TRAIN_MANIFEST)

print()
print("TOKENIZER_PATH")
print(TOKENIZER_PATH)

print()
print("INDEX_PATH")
print(INDEX_PATH)

print()
print("=" * 70)
print("✓ ARTIFACT RESOLUTION COMPLETE")
print("=" * 70)

In [ ]:
TRAIN_MANIFEST = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/"
    "manifests/train_manifest.jsonl"
)

VAL_MANIFEST = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/"
    "manifests/validation_manifest.jsonl"
)

TEST_MANIFEST = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/"
    "manifests/test_manifest.jsonl"
)

TOKENIZER_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/"
    "tokenizer/legal_bpe_32k.json"
)

INDEX_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-final-recovery/"
    "indexes/source_index.jsonl"
)

In [ ]:
# ============================================================
# MAMBA — DATASET CONSISTENCY AUDIT
# READ-ONLY
# ============================================================

import os
import json
import hashlib

print("=" * 70)
print("MAMBA — DATASET CONSISTENCY AUDIT")
print("READ-ONLY — NOTHING WILL BE MODIFIED")
print("=" * 70)


def sha256_file(path, chunk_size=8 * 1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# VERIFY FILES
# ------------------------------------------------------------

required_files = {
    "train_manifest": TRAIN_MANIFEST,
    "validation_manifest": VAL_MANIFEST,
    "test_manifest": TEST_MANIFEST,
    "tokenizer": TOKENIZER_PATH,
    "source_index": INDEX_PATH,
}

for name, path in required_files.items():

    print(f"{name:22s}: {path}")

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"\nMissing required artifact:\n{path}"
        )

    print(
        f"{'':22s}  ✓ exists "
        f"({os.path.getsize(path) / 1024**2:.2f} MB)"
    )


# ------------------------------------------------------------
# MANIFEST RECORD COUNTS
# ------------------------------------------------------------

print()
print("=" * 70)
print("MANIFEST COUNTS")
print("=" * 70)


def count_jsonl(path):

    count = 0

    with open(path, "r") as f:

        for line in f:

            if line.strip():
                count += 1

    return count


train_count = count_jsonl(TRAIN_MANIFEST)
val_count = count_jsonl(VAL_MANIFEST)
test_count = count_jsonl(TEST_MANIFEST)

print("Training examples   :", train_count)
print("Validation examples :", val_count)
print("Test examples       :", test_count)


# ------------------------------------------------------------
# FIRST RECORDS
# ------------------------------------------------------------

print()
print("=" * 70)
print("FIRST TRAIN MANIFEST RECORD")
print("=" * 70)

with open(TRAIN_MANIFEST, "r") as f:

    first_record = json.loads(
        f.readline()
    )

print(
    json.dumps(
        first_record,
        indent=2,
    )
)


# ------------------------------------------------------------
# CHECKSUMS
# ------------------------------------------------------------

print()
print("=" * 70)
print("ARTIFACT CHECKSUMS")
print("=" * 70)

for name, path in required_files.items():

    print(
        f"{name:22s}: "
        f"{sha256_file(path)}"
    )


print()
print("=" * 70)
print("✓ DATASET CONSISTENCY AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — STREAMING CONFIGURATION AUDIT
# READ-ONLY
# ============================================================

import json

STREAMING_CONFIG_PATH = os.path.join(
    TRAINING_PREP_ROOT,
    "streaming_config.json",
)

TRAINING_EXAMPLE_CONFIG_PATH = os.path.join(
    TRAINING_PREP_ROOT,
    "training_example_config.json",
)

print("=" * 70)
print("MAMBA — STREAMING CONFIGURATION")
print("=" * 70)


for name, path in {
    "streaming_config": STREAMING_CONFIG_PATH,
    "training_example_config":
        TRAINING_EXAMPLE_CONFIG_PATH,
}.items():

    print()
    print(name)
    print(path)

    if not os.path.isfile(path):

        print("⚠ NOT FOUND")
        continue

    with open(path, "r") as f:

        config = json.load(f)

    print(
        json.dumps(
            config,
            indent=2,
        )
    )


print()
print("=" * 70)
print("✓ STREAMING CONFIGURATION AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — RECOVERY CONFIGURATION AUDIT
# READ-ONLY
# ============================================================

RECOVERY_MANIFEST = os.path.join(
    FINAL_RECOVERY_ROOT,
    "RECOVERY_MANIFEST.json",
)

RECOVERY_CONFIG = os.path.join(
    FINAL_RECOVERY_ROOT,
    "training",
    "phase4_recovery_config.json",
)

TRAINING_STATE = os.path.join(
    FINAL_RECOVERY_ROOT,
    "training_state.json",
)

MODEL_ARCHITECTURE = os.path.join(
    FINAL_RECOVERY_ROOT,
    "model_architecture.json",
)

print("=" * 70)
print("MAMBA — EXISTING RECOVERY METADATA")
print("=" * 70)


for name, path in {
    "RECOVERY_MANIFEST": RECOVERY_MANIFEST,
    "RECOVERY_CONFIG": RECOVERY_CONFIG,
    "TRAINING_STATE": TRAINING_STATE,
    "MODEL_ARCHITECTURE": MODEL_ARCHITECTURE,
}.items():

    print()
    print("-" * 70)
    print(name)
    print(path)

    if not os.path.isfile(path):

        print("⚠ NOT FOUND")
        continue

    try:

        with open(path, "r") as f:

            data = json.load(f)

        print(
            json.dumps(
                data,
                indent=2,
            )
        )

    except Exception as e:

        print(
            "⚠ Could not parse as JSON:",
            repr(e)
        )


print()
print("=" * 70)
print("✓ RECOVERY METADATA AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — TOKENIZER AUDIT
# READ-ONLY
# ============================================================

import json
import os

print("=" * 70)
print("MAMBA — TOKENIZER AUDIT")
print("=" * 70)

with open(TOKENIZER_PATH, "r") as f:

    tokenizer_json = json.load(f)

print("Tokenizer file:", TOKENIZER_PATH)

print()
print("Top-level keys:")
print(
    list(tokenizer_json.keys())
)


# ------------------------------------------------------------
# VOCABULARY
# ------------------------------------------------------------

vocab = (
    tokenizer_json
    .get("model", {})
    .get("vocab", {})
)

if not vocab:

    raise RuntimeError(
        "Could not locate tokenizer vocabulary."
    )

VOCAB_SIZE = len(vocab)

print()
print("Vocabulary size:", VOCAB_SIZE)


# ------------------------------------------------------------
# SPECIAL TOKENS
# ------------------------------------------------------------

added_tokens = tokenizer_json.get(
    "added_tokens",
    [],
)

print()
print("Added/special tokens:")

for token in added_tokens:

    print(token)


print()
print("=" * 70)
print("✓ TOKENIZER AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA EXPERIMENT — DEPENDENCY / ENVIRONMENT AUDIT
# READ-ONLY — NOTHING WILL BE MODIFIED
# ============================================================

import sys
import os
import importlib
import subprocess
import torch

print("=" * 70)
print("MAMBA — DEPENDENCY / ENVIRONMENT AUDIT")
print("READ-ONLY")
print("=" * 70)

# ============================================================
# PYTHON / PYTORCH / CUDA
# ============================================================

print()
print("=" * 70)
print("PYTHON / PYTORCH / CUDA")
print("=" * 70)

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(
        "GPU memory:",
        f"{props.total_memory / 1024**3:.2f} GB"
    )

assert torch.cuda.is_available(), \
    "CUDA is not available."

# ============================================================
# MAMBA PACKAGE DISCOVERY
# ============================================================

print()
print("=" * 70)
print("MAMBA PACKAGE DISCOVERY")
print("=" * 70)

candidate_packages = [
    "mamba_ssm",
    "causal_conv1d",
]

package_status = {}

for package_name in candidate_packages:

    try:
        module = importlib.import_module(package_name)

        package_status[package_name] = True

        print(
            f"✓ {package_name:20s} "
            f"importable"
        )

        version = getattr(
            module,
            "__version__",
            None
        )

        if version is not None:
            print(
                f"  version: {version}"
            )

        module_file = getattr(
            module,
            "__file__",
            None
        )

        if module_file:
            print(
                f"  path   : {module_file}"
            )

    except Exception as e:

        package_status[package_name] = False

        print(
            f"✗ {package_name:20s} "
            f"not importable"
        )

        print(
            f"  reason: {type(e).__name__}: {e}"
        )

# ============================================================
# PYTORCH 2.x FEATURES NEEDED BY MODERN MAMBA IMPLEMENTATIONS
# ============================================================

print()
print("=" * 70)
print("PYTORCH COMPATIBILITY")
print("=" * 70)

torch_version = tuple(
    int(x)
    for x in torch.__version__.split("+")[0].split(".")[:2]
)

print(
    "PyTorch major/minor:",
    torch_version
)

assert torch_version >= (2, 0), (
    "Mamba experiment requires PyTorch 2.x."
)

print("✓ PyTorch 2.x detected")

# ============================================================
# BASIC CUDA COMPUTATION
# ============================================================

print()
print("=" * 70)
print("CUDA COMPUTATION TEST")
print("=" * 70)

device = torch.device("cuda:0")

x = torch.randn(
    2,
    16,
    768,
    device=device,
    dtype=torch.float32,
)

y = torch.randn(
    2,
    16,
    768,
    device=device,
    dtype=torch.float32,
)

z = torch.matmul(
    x,
    y.transpose(-1, -2),
)

assert z.device == device
assert torch.isfinite(z).all()

print("✓ CUDA tensor allocation")
print("✓ Float32 computation")
print("✓ Matrix multiplication")
print("✓ Finite output")

del x
del y
del z

torch.cuda.empty_cache()

# ============================================================
# OPTIONAL MAMBA API INSPECTION
# ============================================================

print()
print("=" * 70)
print("MAMBA API INSPECTION")
print("=" * 70)

if package_status.get("mamba_ssm", False):

    import mamba_ssm

    print(
        "mamba_ssm location:",
        mamba_ssm.__file__
    )

    try:

        from mamba_ssm import Mamba

        print("✓ from mamba_ssm import Mamba")

        print()
        print("Mamba object:")
        print(Mamba)

    except Exception as e:

        print(
            "✗ Standard Mamba import failed:"
        )

        print(
            type(e).__name__,
            ":",
            e
        )

else:

    print(
        "Mamba package is not currently installed."
    )

# ============================================================
# CAUSAL-CONV API INSPECTION
# ============================================================

print()
print("=" * 70)
print("CAUSAL CONVOLUTION API")
print("=" * 70)

if package_status.get("causal_conv1d", False):

    try:

        import causal_conv1d

        print(
            "✓ causal_conv1d import successful"
        )

        print(
            "Location:",
            causal_conv1d.__file__
        )

    except Exception as e:

        print(
            "✗ causal_conv1d import failed:",
            e
        )

else:

    print(
        "causal_conv1d is not currently installed."
    )

# ============================================================
# MEMORY BASELINE
# ============================================================

print()
print("=" * 70)
print("GPU MEMORY BASELINE")
print("=" * 70)

allocated = (
    torch.cuda.memory_allocated(device)
    / 1024**3
)

reserved = (
    torch.cuda.memory_reserved(device)
    / 1024**3
)

print(
    f"Allocated : {allocated:.3f} GB"
)

print(
    f"Reserved  : {reserved:.3f} GB"
)

# ============================================================
# FINAL VERDICT
# ============================================================

print()
print("=" * 70)
print("ENVIRONMENT AUDIT VERDICT")
print("=" * 70)

if package_status.get("mamba_ssm", False):

    print(
        "✓ mamba_ssm is already available."
    )

    print(
        "✓ Environment is ready for Mamba API inspection."
    )

else:

    print(
        "⚠ mamba_ssm is NOT installed."
    )

    print(
        "No installation was attempted."
    )

    print(
        "Do NOT start Mamba training yet."
    )

print()
print(
    "✓ READ-ONLY AUDIT COMPLETE"
)

print(
    "✓ NO CHECKPOINTS MODIFIED"
)

print(
    "✓ NO DATASETS MODIFIED"
)

print(
    "✓ NO TRAINING STARTED"
)

print("=" * 70)

In [ ]:
# ============================================================
# MAMBA EXPERIMENT — DEPENDENCY INSTALLATION
# INSTALL ONLY — NO DATA / CHECKPOINT MODIFICATION
# ============================================================

import sys
import subprocess

print("=" * 70)
print("MAMBA — INSTALLATION")
print("=" * 70)

print("Python:", sys.version)

# Install causal-conv1d first.
# --no-build-isolation is intentional: keep the existing
# Kaggle CUDA-enabled PyTorch environment.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-build-isolation",
    "causal-conv1d>=1.4.0",
])

# Install Mamba.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-build-isolation",
    "mamba-ssm",
])

print()
print("=" * 70)
print("✓ MAMBA INSTALLATION COMMANDS COMPLETED")
print("=" * 70)
print("IMPORTANT:")
print("Run the next audit cell before constructing any model.")

In [ ]:
# ============================================================
# MAMBA — POST-INSTALLATION AUDIT
# READ-ONLY
# ============================================================

import sys
import torch

print("=" * 70)
print("MAMBA — POST-INSTALLATION AUDIT")
print("=" * 70)

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(0))

print()
print("PACKAGE IMPORT")
print("-" * 70)

mamba_ok = False
conv_ok = False

try:
    import mamba_ssm
    from mamba_ssm import Mamba
    mamba_ok = True

    print("✓ mamba_ssm imported")
    print("  Version:", getattr(mamba_ssm, "__version__", "unknown"))
    print("  Mamba class:", Mamba)

except Exception as e:
    print("✗ mamba_ssm import failed")
    print(" ", type(e).__name__, ":", e)

try:
    import causal_conv1d
    conv_ok = True

    print("✓ causal_conv1d imported")
    print(
        "  Version:",
        getattr(causal_conv1d, "__version__", "unknown")
    )

except Exception as e:
    print("✗ causal_conv1d import failed")
    print(" ", type(e).__name__, ":", e)

print()
print("CUDA")
print("-" * 70)

assert torch.cuda.is_available()

x = torch.randn(
    2,
    32,
    768,
    device="cuda",
    dtype=torch.float32,
)

print("✓ CUDA tensor created")
print("✓ Tensor shape:", tuple(x.shape))
print("✓ Tensor dtype:", x.dtype)
print("✓ Tensor device:", x.device)

print()
print("=" * 70)

if mamba_ok and conv_ok:
    print("✓ DEPENDENCY AUDIT PASSED")
    print("✓ Mamba can now be tested.")
else:
    print("✗ DEPENDENCY AUDIT FAILED")
    print("DO NOT CONTINUE TO MODEL CONSTRUCTION.")

print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — CUDA FORWARD/BACKWARD SMOKE TEST
# READ-ONLY WITH RESPECT TO DATA/CHECKPOINTS
# ============================================================

import gc
import torch

from mamba_ssm import Mamba

print("=" * 70)
print("MAMBA — CUDA SMOKE TEST")
print("=" * 70)

DEVICE = torch.device("cuda:0")

# Small test first.
BATCH = 1
SEQ_LEN = 128
D_MODEL = 768

print("Device :", DEVICE)
print("Batch  :", BATCH)
print("Seq    :", SEQ_LEN)
print("Dim    :", D_MODEL)

model = Mamba(
    d_model=D_MODEL,
    d_state=16,
    d_conv=4,
    expand=2,
).to(DEVICE)

model.train()

print()
print("Model parameters:",
      sum(p.numel() for p in model.parameters()))

x = torch.randn(
    BATCH,
    SEQ_LEN,
    D_MODEL,
    device=DEVICE,
    dtype=torch.float32,
    requires_grad=True,
)

print("✓ Input created")

y = model(x)

assert y.shape == x.shape
assert torch.isfinite(y).all()

print("✓ Forward pass successful")
print("Output shape:", tuple(y.shape))

loss = y.float().square().mean()

assert torch.isfinite(loss)

loss.backward()

print("✓ Backward pass successful")
print("Loss:", float(loss))

has_grad = any(
    p.grad is not None
    for p in model.parameters()
)

assert has_grad

print("✓ Gradients present")

print()
print("GPU allocated:",
      f"{torch.cuda.memory_allocated(DEVICE)/1024**3:.3f} GB")

print("GPU reserved :",
      f"{torch.cuda.memory_reserved(DEVICE)/1024**3:.3f} GB")

del x
del y
del loss
del model

gc.collect()
torch.cuda.empty_cache()

print()
print("=" * 70)
print("✓ MAMBA CUDA SMOKE TEST PASSED")
print("✓ NO DATASET MODIFIED")
print("✓ NO CHECKPOINT MODIFIED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA EXPERIMENT — LANGUAGE MODEL ARCHITECTURE
# NO DATASET / CHECKPOINT MODIFICATION
# ============================================================

import gc
import torch
import torch.nn as nn

from mamba_ssm import Mamba

# ============================================================
# CANONICAL EXPERIMENT CONFIG
# ============================================================

DEVICE = torch.device("cuda:0")

VOCAB_SIZE = 32000
CONTEXT_LENGTH = 4096

# Match the existing Phase-4 Transformer width/depth.
N_LAYER = 12
D_MODEL = 768

# Mamba-specific configuration.
D_STATE = 16
D_CONV = 4
EXPAND = 2

# ============================================================
# MAMBA LANGUAGE MODEL
# ============================================================

class MambaBlock(nn.Module):

    def __init__(
        self,
        d_model,
        d_state,
        d_conv,
        expand,
    ):
        super().__init__()

        self.norm = nn.RMSNorm(d_model)

        self.mixer = Mamba(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand,
        )

    def forward(self, x):
        return x + self.mixer(self.norm(x))


class MambaLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layer,
        d_state,
        d_conv,
        expand,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layer = n_layer

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
        )

        self.layers = nn.ModuleList([
            MambaBlock(
                d_model=d_model,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
            )
            for _ in range(n_layer)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        # Weight tying — same token embedding and LM head weights.
        self.lm_head.weight = self.token_embedding.weight

    def forward(
        self,
        input_ids,
        labels=None,
    ):

        x = self.token_embedding(input_ids)

        for layer in self.layers:
            x = layer(x)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        loss = None

        if labels is not None:

            loss = nn.functional.cross_entropy(
                logits.reshape(-1, self.vocab_size),
                labels.reshape(-1),
                ignore_index=-100,
            )

        return {
            "logits": logits,
            "loss": loss,
        }


# ============================================================
# CONSTRUCT
# ============================================================

mamba_model = MambaLanguageModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layer=N_LAYER,
    d_state=D_STATE,
    d_conv=D_CONV,
    expand=EXPAND,
).to(DEVICE)

print("=" * 70)
print("MAMBA LANGUAGE MODEL CREATED")
print("=" * 70)

print("Device          :", DEVICE)
print("Vocabulary      :", VOCAB_SIZE)
print("Context         :", CONTEXT_LENGTH)
print("Layers          :", N_LAYER)
print("Model dimension :", D_MODEL)
print("d_state         :", D_STATE)
print("d_conv          :", D_CONV)
print("expand          :", EXPAND)

In [ ]:
# ============================================================
# MAMBA — ARCHITECTURE AUDIT
# READ-ONLY
# ============================================================

print("=" * 70)
print("MAMBA — ARCHITECTURE AUDIT")
print("=" * 70)

parameter_count = sum(
    p.numel()
    for p in mamba_model.parameters()
)

trainable_count = sum(
    p.numel()
    for p in mamba_model.parameters()
    if p.requires_grad
)

print()
print("TOTAL PARAMETERS")
print("-" * 70)
print(f"{parameter_count:,}")

print()
print("TRAINABLE PARAMETERS")
print("-" * 70)
print(f"{trainable_count:,}")

print()
print("REFERENCE TRANSFORMER")
print("-" * 70)
print("Original Phase-4 Transformer : 112,777,728 parameters")
print("Mamba parameter count        :", f"{parameter_count:,}")

print()
print("PARAMETER RATIO")
print("-" * 70)
print(
    "Mamba / Transformer:",
    f"{parameter_count / 112777728:.4f}"
)

print()
print("MODEL COMPONENTS")
print("-" * 70)

for name, parameter in mamba_model.named_parameters():

    print(
        f"{name:<55}",
        f"{tuple(parameter.shape)!s:<25}",
        f"{parameter.numel():>12,}"
    )

print()
print("=" * 70)
print("✓ ARCHITECTURE AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — 4096 TOKEN MEMORY / FORWARD / BACKWARD TEST
# READ-ONLY WITH RESPECT TO DATA AND CHECKPOINTS
# ============================================================

import gc
import torch

print("=" * 70)
print("MAMBA — 4096 TOKEN GPU TEST")
print("=" * 70)

mamba_model.train()

torch.cuda.empty_cache()
gc.collect()

torch.cuda.reset_peak_memory_stats(DEVICE)

BATCH_SIZE_TEST = 1

input_ids = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        BATCH_SIZE_TEST,
        CONTEXT_LENGTH,
    ),
    device=DEVICE,
    dtype=torch.long,
)

labels = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        BATCH_SIZE_TEST,
        CONTEXT_LENGTH,
    ),
    device=DEVICE,
    dtype=torch.long,
)

print("Input shape :", tuple(input_ids.shape))
print("Context     :", CONTEXT_LENGTH)

# ------------------------------------------------------------
# FORWARD
# ------------------------------------------------------------

outputs = mamba_model(
    input_ids=input_ids,
    labels=labels,
)

loss = outputs["loss"]

assert loss is not None
assert torch.isfinite(loss)

print()
print("✓ Forward pass successful")
print("Loss:", loss.detach().item())

# ------------------------------------------------------------
# BACKWARD
# ------------------------------------------------------------

loss.backward()

print("✓ Backward pass successful")

# ------------------------------------------------------------
# GRADIENT CHECK
# ------------------------------------------------------------

gradient_count = sum(
    1
    for p in mamba_model.parameters()
    if p.grad is not None
)

print("Parameters with gradients:", gradient_count)

assert gradient_count > 0

# ------------------------------------------------------------
# MEMORY
# ------------------------------------------------------------

allocated = (
    torch.cuda.memory_allocated(DEVICE)
    / 1024**3
)

reserved = (
    torch.cuda.memory_reserved(DEVICE)
    / 1024**3
)

peak_allocated = (
    torch.cuda.max_memory_allocated(DEVICE)
    / 1024**3
)

peak_reserved = (
    torch.cuda.max_memory_reserved(DEVICE)
    / 1024**3
)

print()
print("GPU MEMORY")
print("-" * 70)
print(f"Allocated       : {allocated:.3f} GB")
print(f"Reserved        : {reserved:.3f} GB")
print(f"Peak allocated  : {peak_allocated:.3f} GB")
print(f"Peak reserved   : {peak_reserved:.3f} GB")

# ------------------------------------------------------------
# CLEANUP
# ------------------------------------------------------------

del input_ids
del labels
del outputs
del loss

mamba_model.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()

print()
print("=" * 70)
print("✓ 4096-TOKEN MAMBA TEST COMPLETE")
print("✓ NO DATASET MODIFIED")
print("✓ NO CHECKPOINT MODIFIED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — PRE-TRAINING SAFETY GATE
# ============================================================

print("=" * 70)
print("MAMBA — PRE-TRAINING SAFETY GATE")
print("=" * 70)

free_bytes, total_bytes = torch.cuda.mem_get_info(DEVICE)

free_gb = free_bytes / 1024**3
total_gb = total_bytes / 1024**3

print("GPU total memory :", f"{total_gb:.2f} GB")
print("GPU free memory  :", f"{free_gb:.2f} GB")

print()
print("REQUIRED CONDITIONS")
print("-" * 70)

checks = {
    "CUDA available":
        torch.cuda.is_available(),

    "Correct vocabulary":
        VOCAB_SIZE == 32000,

    "Correct context":
        CONTEXT_LENGTH == 4096,

    "Correct depth":
        N_LAYER == 12,

    "Correct width":
        D_MODEL == 768,

    "Model on CUDA":
        next(mamba_model.parameters()).device.type == "cuda",

    "Parameters exist":
        parameter_count > 0,

    "4096-token test completed":
        True,

    "No checkpoint loaded":
        True,
}

all_pass = True

for name, result in checks.items():

    print(
        ("✓ " if result else "✗ ") + name
    )

    if not result:
        all_pass = False

print()

if all_pass:

    print("=" * 70)
    print("✓ MAMBA ARCHITECTURE PASSED PRE-TRAINING GATE")
    print("=" * 70)
    print()
    print("NEXT STEP:")
    print("Integrate the existing document-aware streaming dataset.")
    print("Do NOT create the training loop yet.")

else:

    raise RuntimeError(
        "Mamba architecture did not pass the pre-training gate."
    )

In [ ]:
# ============================================================
# MAMBA — STREAMING PIPELINE INTEGRATION AUDIT
# READ-ONLY
# ============================================================

import inspect
import torch

print("=" * 70)
print("MAMBA — STREAMING PIPELINE INTEGRATION AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# REQUIRED OBJECTS
# ------------------------------------------------------------

required_objects = [
    "TRAIN_MANIFEST",
    "TOKENIZER_PATH",
]

print()
print("CANONICAL ARTIFACTS")
print("-" * 70)

for name in required_objects:

    exists = name in globals()

    print(
        ("✓ " if exists else "✗ "),
        name,
        "=", globals().get(name, "<missing>")
    )

    if not exists:
        raise RuntimeError(
            f"Required artifact variable missing: {name}"
        )

# ------------------------------------------------------------
# DATASET CLASS
# ------------------------------------------------------------

print()
print("DATASET CLASS")
print("-" * 70)

dataset_class = globals().get(
    "WorkerSafeDocumentAwarePacker"
)

if dataset_class is None:

    raise RuntimeError(
        "WorkerSafeDocumentAwarePacker is not defined."
    )

print("Class:", dataset_class)
print("Module:", dataset_class.__module__)

print()
print("Constructor signature:")
print(
    inspect.signature(
        dataset_class.__init__
    )
)

# ------------------------------------------------------------
# EXISTING TRAIN LOADER
# ------------------------------------------------------------

print()
print("TRAIN LOADER")
print("-" * 70)

if "train_loader" not in globals():

    raise RuntimeError(
        "Existing train_loader is not available."
    )

print("Type:", type(train_loader))
print("Dataset:", type(train_loader.dataset))
print("Batch size:", train_loader.batch_size)
print("Drop last:", train_loader.drop_last)
print("Workers:", train_loader.num_workers)

# ------------------------------------------------------------
# DATASET ATTRIBUTES
# ------------------------------------------------------------

dataset = train_loader.dataset

print()
print("DATASET ATTRIBUTES")
print("-" * 70)

for attr in [
    "index",
    "context_length",
    "seed",
    "document_boundary",
    "packing",
]:

    if hasattr(dataset, attr):

        print(
            f"{attr:<22}:",
            getattr(dataset, attr)
        )

    else:

        print(
            f"{attr:<22}: <not exposed>"
        )

# ------------------------------------------------------------
# IMPORTANT — DO NOT CREATE A NEW DATASET YET
# ------------------------------------------------------------

print()
print("=" * 70)
print("✓ EXISTING STREAMING PIPELINE FOUND")
print("=" * 70)

print()
print("IMPORTANT:")
print("We will reuse the existing DataLoader.")
print("No new dataset will be constructed.")
print("No manifest will be rewritten.")
print("No tokenized corpus will be materialized.")
print("No checkpoint will be loaded.")

print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — CANONICAL STREAMING CONFIGURATION
# ============================================================

import os
import json
import gc
import torch

from pathlib import Path
from tokenizers import Tokenizer
from torch.utils.data import DataLoader, IterableDataset, get_worker_info


# ============================================================
# KAGGLE DATASET ROOTS
# ============================================================

TRAINING_PREP_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

FINAL_RECOVERY_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-final-recovery"
)


# ============================================================
# CANONICAL ARTIFACTS
# ============================================================

TRAIN_MANIFEST = (
    TRAINING_PREP_ROOT
    / "manifests"
    / "train_manifest.jsonl"
)

VAL_MANIFEST = (
    TRAINING_PREP_ROOT
    / "manifests"
    / "validation_manifest.jsonl"
)

TEST_MANIFEST = (
    TRAINING_PREP_ROOT
    / "manifests"
    / "test_manifest.jsonl"
)

TOKENIZER_PATH = (
    TRAINING_PREP_ROOT
    / "tokenizer"
    / "legal_bpe_32k.json"
)

INDEX_PATH = (
    FINAL_RECOVERY_ROOT
    / "indexes"
    / "source_index.jsonl"
)

CONTEXT_LENGTH = 4096
BATCH_SIZE = 1
SEED = 42


# ============================================================
# VALIDATE
# ============================================================

print("=" * 70)
print("MAMBA — CANONICAL STREAMING CONFIGURATION")
print("=" * 70)

paths = {
    "Training prep": TRAINING_PREP_ROOT,
    "Final recovery": FINAL_RECOVERY_ROOT,
    "Train manifest": TRAIN_MANIFEST,
    "Validation manifest": VAL_MANIFEST,
    "Test manifest": TEST_MANIFEST,
    "Tokenizer": TOKENIZER_PATH,
    "Source index": INDEX_PATH,
}

for name, path in paths.items():

    print(
        f"{name:<22}: "
        f"{'✓' if path.exists() else '✗'} "
        f"{path}"
    )

    assert path.exists(), f"Missing: {path}"


print()
print("Context length:", CONTEXT_LENGTH)
print("Batch size    :", BATCH_SIZE)
print("Seed          :", SEED)

print()
print("=" * 70)
print("✓ CANONICAL STREAMING ARTIFACTS VERIFIED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — LOAD CANONICAL TOKENIZER
# ============================================================

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

VOCAB_SIZE = tokenizer.get_vocab_size()

print("=" * 70)
print("MAMBA — TOKENIZER")
print("=" * 70)

print("Path       :", TOKENIZER_PATH)
print("Vocabulary :", VOCAB_SIZE)

assert VOCAB_SIZE == 32000

for token in [
    "[PAD]",
    "[UNK]",
    "[BOS]",
    "[EOS]",
]:

    token_id = tokenizer.token_to_id(token)

    print(
        f"{token:<8}: {token_id}"
    )

    assert token_id is not None

print()
print("✓ Canonical 32K tokenizer loaded")

In [ ]:
# ============================================================
# MAMBA — LOAD CANONICAL TOKENIZER
# ============================================================

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

VOCAB_SIZE = tokenizer.get_vocab_size()

print("=" * 70)
print("MAMBA — TOKENIZER")
print("=" * 70)

print("Path       :", TOKENIZER_PATH)
print("Vocabulary :", VOCAB_SIZE)

assert VOCAB_SIZE == 32000

for token in [
    "[PAD]",
    "[UNK]",
    "[BOS]",
    "[EOS]",
]:

    token_id = tokenizer.token_to_id(token)

    print(
        f"{token:<8}: {token_id}"
    )

    assert token_id is not None

print()
print("✓ Canonical 32K tokenizer loaded")

In [ ]:
# ============================================================
# MAMBA — DOCUMENT-AWARE STREAMING PACKER
# SAME IMPLEMENTATION AS PHASE 4
# ============================================================

import os
import json
import torch

from torch.utils.data import IterableDataset


class DocumentAwarePacker(IterableDataset):

    def __init__(
        self,
        manifest_path,
        index_path,
        tokenizer,
        context_length=4096,
        checkpoint=str(TRAINING_PREP_ROOT),
        split="train",
    ):
        super().__init__()

        self.manifest_path = str(manifest_path)
        self.index_path = str(index_path)
        self.tokenizer = tokenizer
        self.context_length = int(context_length)
        self.checkpoint = str(checkpoint)
        self.split = split

        # ----------------------------------------------------
        # LOAD BYTE-OFFSET INDEX
        # ----------------------------------------------------

        self.index = {}

        with open(
            self.index_path,
            "r",
            encoding="utf-8",
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                rec = json.loads(line)

                self.index[
                    rec["text_hash"]
                ] = {
                    "source_file":
                        rec["source_file"],

                    "byte_offset":
                        rec["byte_offset"],

                    "byte_length":
                        rec["byte_length"],
                }

        print(
            f"Loaded index entries: "
            f"{len(self.index):,}"
        )

        # ----------------------------------------------------
        # LOAD MANIFEST
        # ----------------------------------------------------

        self.manifest = []

        with open(
            self.manifest_path,
            "r",
            encoding="utf-8",
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                rec = json.loads(line)

                self.manifest.append(rec)

        print(
            f"Loaded manifest records: "
            f"{len(self.manifest):,}"
        )

        # ----------------------------------------------------
        # SPECIAL TOKENS
        # ----------------------------------------------------

        self.bos_id = self._get_token_id("[BOS]")
        self.eos_id = self._get_token_id("[EOS]")
        self.pad_id = self._get_token_id("[PAD]")

        print("BOS id :", self.bos_id)
        print("EOS id :", self.eos_id)
        print("PAD id :", self.pad_id)

    # --------------------------------------------------------
    # TOKENIZER COMPATIBILITY
    # --------------------------------------------------------

    def _get_token_id(self, token):

        token_id = None

        if hasattr(
            self.tokenizer,
            "token_to_id",
        ):

            token_id = self.tokenizer.token_to_id(
                token
            )

        elif hasattr(
            self.tokenizer,
            "convert_tokens_to_ids",
        ):

            token_id = (
                self.tokenizer
                .convert_tokens_to_ids(token)
            )

        if token_id is None:

            raise ValueError(
                "Tokenizer does not contain "
                f"required special token: {token}"
            )

        if isinstance(token_id, list):

            if len(token_id) != 1:

                raise ValueError(
                    f"Unexpected token id for "
                    f"{token}: {token_id}"
                )

            token_id = token_id[0]

        return int(token_id)

    # --------------------------------------------------------
    # RESOLVE SOURCE DOCUMENT
    # --------------------------------------------------------

    def _read_source_record(self, rec):

        text_hash = rec["text_hash"]

        if text_hash not in self.index:

            raise KeyError(
                "text_hash not found in "
                f"source index: {text_hash}"
            )

        idx = self.index[text_hash]

        relative_source = idx["source_file"]

        source_name = relative_source

        if source_name.endswith(".gz"):

            source_name = source_name[:-3]

        source_path = os.path.join(
            self.checkpoint,
            "dedup",
            source_name,
        )

        if not os.path.exists(source_path):

            raise FileNotFoundError(
                "Resolved source does not exist:\n"
                f"{source_path}"
            )

        with open(
            source_path,
            "rb",
        ) as f:

            f.seek(idx["byte_offset"])

            raw = f.read(
                idx["byte_length"]
            )

        raw = raw.rstrip(
            b"\r\n"
        )

        source_record = json.loads(
            raw.decode("utf-8")
        )

        return source_record

    # --------------------------------------------------------
    # EXTRACT TEXT
    # --------------------------------------------------------

    def _extract_text(self, source_record):

        if isinstance(
            source_record,
            str,
        ):

            return source_record

        for key in (
            "text",
            "content",
            "document",
            "body",
        ):

            value = source_record.get(key)

            if isinstance(
                value,
                str,
            ):

                return value

        raise KeyError(
            "Could not find document text field. "
            f"Available fields: "
            f"{list(source_record.keys())}"
        )

    # --------------------------------------------------------
    # TOKENIZE DOCUMENT
    # --------------------------------------------------------

    def _tokenize_document(self, text):

        if hasattr(
            self.tokenizer,
            "encode",
        ):

            encoded = self.tokenizer.encode(
                text
            )

            if hasattr(
                encoded,
                "ids",
            ):

                token_ids = encoded.ids

            elif isinstance(
                encoded,
                dict,
            ):

                token_ids = encoded[
                    "input_ids"
                ]

            else:

                token_ids = encoded

        else:

            raise TypeError(
                "Unsupported tokenizer interface."
            )

        token_ids = list(
            map(int, token_ids)
        )

        # Explicit document boundaries.
        token_ids = (
            [self.bos_id]
            + token_ids
            + [self.eos_id]
        )

        return token_ids

    # --------------------------------------------------------
    # DOCUMENT-AWARE PACKING
    # --------------------------------------------------------

    def _pack_document(self, token_ids):

        if len(token_ids) > self.context_length:

            for start in range(
                0,
                len(token_ids),
                self.context_length,
            ):

                chunk = token_ids[
                    start:
                    start + self.context_length
                ]

                if len(chunk) < self.context_length:

                    chunk = (
                        chunk
                        + [self.pad_id]
                        * (
                            self.context_length
                            - len(chunk)
                        )
                    )

                yield chunk

            return

        chunk = token_ids[:]

        if len(chunk) < self.context_length:

            chunk = (
                chunk
                + [self.pad_id]
                * (
                    self.context_length
                    - len(chunk)
                )
            )

        yield chunk

    # --------------------------------------------------------
    # STREAM
    # --------------------------------------------------------

    def __iter__(self):

        for rec in self.manifest:

            token_ids = self._tokenize_document(
                self._extract_text(
                    self._read_source_record(rec)
                )
            )

            for packed in self._pack_document(
                token_ids
            ):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long,
                )

                labels = input_ids.clone()

                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids":
                        input_ids,

                    "labels":
                        labels,

                    "attention_mask":
                        attention_mask,
                }


print("=" * 70)
print("✓ DOCUMENT-AWARE PACKER DEFINED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — WORKER-SAFE STREAMING DATASET
# SAME PHASE-4 IMPLEMENTATION
# ============================================================

class WorkerSafeDocumentAwarePacker(
    DocumentAwarePacker
):

    def __iter__(self):

        worker_info = get_worker_info()

        # ----------------------------------------------------
        # SINGLE PROCESS
        # ----------------------------------------------------

        if worker_info is None:

            start = 0
            step = 1

        # ----------------------------------------------------
        # MULTI-WORKER
        # ----------------------------------------------------

        else:

            worker_id = worker_info.id
            num_workers = worker_info.num_workers

            start = worker_id
            step = num_workers

        # ----------------------------------------------------
        # STRIDED MANIFEST PARTITION
        # ----------------------------------------------------

        for manifest_idx in range(
            start,
            len(self.manifest),
            step,
        ):

            rec = self.manifest[
                manifest_idx
            ]

            source_record = (
                self._read_source_record(rec)
            )

            text = self._extract_text(
                source_record
            )

            token_ids = (
                self._tokenize_document(text)
            )

            for packed in self._pack_document(
                token_ids
            ):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long,
                )

                labels = input_ids.clone()

                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids":
                        input_ids,

                    "labels":
                        labels,

                    "attention_mask":
                        attention_mask,
                }


print("=" * 70)
print("✓ WORKER-SAFE PACKER DEFINED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — CREATE STREAMING TRAIN DATASET
# NO TOKENIZED CORPUS CREATED
# ============================================================

gc.collect()

train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=str(TRAINING_PREP_ROOT),
    split="train",
)

print("=" * 70)
print("MAMBA — TRAINING DATASET")
print("=" * 70)

print("Dataset type :", type(train_dataset).__name__)
print("Manifest     :", TRAIN_MANIFEST)
print("Index        :", INDEX_PATH)
print("Tokenizer    :", TOKENIZER_PATH)
print("Context      :", CONTEXT_LENGTH)
print("Split        :", "train")

print()
print("Manifest records:")
print(f"{len(train_dataset.manifest):,}")

print()
print("Index records:")
print(f"{len(train_dataset.index):,}")

assert len(train_dataset.manifest) == 3_020_655

print()
print("✓ Streaming dataset created")
print("✓ No tokenized corpus materialized")

In [ ]:
# ============================================================
# MAMBA — SINGLE-WORKER STREAMING BATCH TEST
# READ-ONLY
# ============================================================

import time

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    num_workers=0,
    pin_memory=True,
)

print("=" * 70)
print("MAMBA — FIRST REAL TRAINING BATCH")
print("=" * 70)

t0 = time.time()

batch = next(iter(train_loader))

elapsed = time.time() - t0

print("Load time:", round(elapsed, 3), "seconds")

print()
print("BATCH")
print("-" * 70)

for key, value in batch.items():

    print(
        f"{key:<18}: "
        f"shape={tuple(value.shape)} "
        f"dtype={value.dtype}"
    )

# ------------------------------------------------------------
# SHAPE CHECKS
# ------------------------------------------------------------

assert batch["input_ids"].shape == (
    1,
    CONTEXT_LENGTH,
)

assert batch["labels"].shape == (
    1,
    CONTEXT_LENGTH,
)

assert batch["attention_mask"].shape == (
    1,
    CONTEXT_LENGTH,
)

# ------------------------------------------------------------
# VALUE CHECKS
# ------------------------------------------------------------

assert batch["input_ids"].dtype == torch.long
assert batch["labels"].dtype == torch.long
assert batch["attention_mask"].dtype == torch.long

assert torch.all(
    (batch["input_ids"] >= 0)
    & (batch["input_ids"] < VOCAB_SIZE)
)

assert torch.all(
    (batch["attention_mask"] == 0)
    | (batch["attention_mask"] == 1)
)

print()
print("✓ Shapes correct")
print("✓ Dtypes correct")
print("✓ Token IDs within vocabulary")
print("✓ Attention mask valid")

print()
print("=" * 70)
print("✓ REAL STREAMING BATCH PASSED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — REAL DATA FORWARD/BACKWARD INTEGRATION TEST
# ============================================================

DEVICE = torch.device("cuda:0")

mamba_model.train()
mamba_model.zero_grad(set_to_none=True)

input_ids = batch["input_ids"].to(
    DEVICE,
    dtype=torch.long,
)

labels = batch["labels"].to(
    DEVICE,
    dtype=torch.long,
)

attention_mask = batch[
    "attention_mask"
].to(
    DEVICE,
    dtype=torch.long,
)

print("=" * 70)
print("MAMBA — REAL STREAMING DATA INTEGRATION")
print("=" * 70)

print("Input shape :", tuple(input_ids.shape))
print("Labels shape:", tuple(labels.shape))

outputs = mamba_model(
    input_ids=input_ids,
    labels=labels,
)

loss = outputs["loss"]

assert loss is not None
assert torch.isfinite(loss)

print()
print("✓ Real batch forward pass successful")
print("Loss:", loss.detach().item())

loss.backward()

print("✓ Real batch backward pass successful")

gradient_count = sum(
    p.grad is not None
    for p in mamba_model.parameters()
)

print(
    "Parameters with gradients:",
    gradient_count,
)

assert gradient_count > 0

print()
print("=" * 70)
print("✓ MAMBA + REAL STREAMING PIPELINE PASSED")
print("=" * 70)
print()
print("STOP HERE.")
print("No optimizer step was performed.")
print("No checkpoint was created.")